# 영공간과 열공간

> 선형대수 6강 · 벡터공간

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [영공간과 열공간](https://mioon1402.github.io/timeseriesdata/linalg/L06-nullspace.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 동차와 비동차

## 1. 영공간 N(A)

## 2. 열공간 C(A)

## 3. 두 공간을 눈으로 보기

## 4. RREF와 특별해 구하기

## 5. 완전해 = 특수해 + 영공간

## 6. numpy 로 확인하기

**6-1. RREF 를 직접 만들어보기**

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

def rref(M, tol=1e-10):
    """기약행사다리꼴과 피봇 열 목록을 돌려준다."""
    R = M.astype(float).copy()
    행수, 열수 = R.shape
    피봇열 = []
    r = 0
    for c in range(열수):
        if r >= 행수:
            break
        # 이 열에서 절댓값이 가장 큰 행을 위로 (수치 안정성 — 3강 참고)
        i = np.argmax(np.abs(R[r:, c])) + r
        if abs(R[i, c]) < tol:
            continue                        # 이 열엔 피봇이 없다 = 자유 열
        R[[r, i]] = R[[i, r]]
        R[r] /= R[r, c]                     # 피봇을 1로
        for j in range(행수):               # 그 열의 나머지를 전부 0으로
            if j != r and abs(R[j, c]) > tol:
                R[j] -= R[j, c] * R[r]
        피봇열.append(c)
        r += 1
    return R, 피봇열

A = np.array([[1., 2., 2., 2.],
              [2., 4., 6., 8.],
              [3., 6., 8., 10.]])

R, 피봇 = rref(A)
print("RREF =")
print(R)
print("\n피봇 열:", 피봇, "  자유 열:", [c for c in range(A.shape[1]) if c not in 피봇])
print("랭크 r =", len(피봇))

**6-2. 특별해로 영공간 구하기**

In [ ]:
def 특별해들(A):
    R, 피봇 = rref(A)
    n = A.shape[1]
    자유 = [c for c in range(n) if c not in 피봇]
    해목록 = []
    for f in 자유:
        x = np.zeros(n)
        x[f] = 1                       # 이 자유 변수만 1
        for i, p in enumerate(피봇):
            x[p] = -R[i, f]            # 나머지는 RREF 가 알려준다
        해목록.append(x)
    return np.array(해목록).T          # 열마다 특별해 하나

N = 특별해들(A)
print("특별해 (열마다 하나) =")
print(N)
print("\nA @ N  (전부 0이어야 함) =")
print(A @ N)
print("\n영공간 차원 =", N.shape[1])

**6-3. 영공간의 아무 벡터나 0으로 간다**

In [ ]:
rng = np.random.default_rng(0)

for _ in range(3):
    가중치 = rng.normal(size=N.shape[1])
    n_vec = N @ 가중치                 # 특별해들의 아무 선형결합
    print(f"x = {np.round(n_vec, 2)}   →   A@x = {np.round(A @ n_vec, 9)}")

print("\n→ 영공간의 어떤 벡터를 넣어도 결과는 0. 이게 '부분공간' 이라는 뜻이다.")

**6-4. 라이브러리로도 확인**

In [ ]:
from scipy.linalg import null_space

N2 = null_space(A)
print("scipy 의 영공간 기저 =")
print(N2)
print("\n모양:", N2.shape, " ← 차원은 우리 것과 같다:", N.shape[1] == N2.shape[1])
print("A @ N2 최대 오차:", np.abs(A @ N2).max())
print()
print("기저 벡터가 서로 달라도 괜찮습니다 —")
print("같은 공간을 서로 다른 축으로 표현한 것뿐입니다 (7강 '기저는 유일하지 않다').")

**6-5. Ax=b 가 풀리는가 — 랭크로 판정**

In [ ]:
def 풀리는가(A, b):
    r  = np.linalg.matrix_rank(A)
    rb = np.linalg.matrix_rank(np.column_stack([A, b]))
    return r, rb, r == rb

b1 = A @ np.array([1., 0., 1., 0.])      # 일부러 열공간 안에서 만든 b
b2 = np.array([1., 0., 0.])              # 아무렇게나 고른 b

for 이름, b in [("열공간 안의 b", b1), ("임의의 b", b2)]:
    r, rb, ok = 풀리는가(A, b)
    print(f"{이름:14s} rank(A)={r}  rank([A|b])={rb}  →  {'풀린다' if ok else '해 없음'}")

print()
print("b 를 붙였는데 랭크가 늘어났다 = b 가 열들의 조합으로 안 만들어진다 = 열공간 밖")

**6-6. 완전해 = 특수해 + 영공간**

In [ ]:
b = b1
xp, *_ = np.linalg.lstsq(A, b, rcond=None)     # 특수해 하나
print("특수해 xp =", np.round(xp, 3))
print("A @ xp    =", np.round(A @ xp, 6), " = b ?", np.allclose(A @ xp, b))
print()

for k in [0.0, 3.7, -12.5]:
    x = xp + k * N[:, 0]                        # 영공간을 아무만큼 더해도
    print(f"xp + {k:>6} · n₁  →  A@x = {np.round(A @ x, 6)}   해인가: {np.allclose(A @ x, b)}")

print()
print("→ 해가 '무한히 많다'. 정확히는 xp 를 지나는 2차원 평면 전체다")
print("   (영공간 차원이 2이므로)")

**6-7. 연습문제**

In [ ]:
# 문제 1. A = [[1, 2], [2, 4]] 의 영공간을 특별해로 구해보세요.

# 문제 2. 그 A 에 대해 b = [3, 6] 과 b = [3, 7] 중 어느 쪽이 풀리나요?
#         랭크로 판정해보세요.

# 문제 3. 열이 선형독립인 행렬(예: [[1,0],[0,1],[1,1]])의 영공간은
#         무엇일까요? 차원은?

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
# 문제 1
A1 = np.array([[1., 2.], [2., 4.]])
N1 = 특별해들(A1)
print("영공간 기저 =\n", N1, "  ← (-2, 1) 방향 직선")
print("A1 @ N1 =", (A1 @ N1).ravel())

# 문제 2
for b in [np.array([3., 6.]), np.array([3., 7.])]:
    r, rb, ok = 풀리는가(A1, b)
    print(f"\nb={b}  rank {r} vs {rb}  →  {'풀린다 (해 무한)' if ok else '해 없음'}")

# 문제 3 — 열이 독립이면 영공간은 원점뿐
A3 = np.array([[1., 0.], [0., 1.], [1., 1.]])
N3 = 특별해들(A3)
print("\n랭크 =", np.linalg.matrix_rank(A3), " 열 수 =", A3.shape[1])
print("영공간 차원 =", N3.shape[1] if N3.size else 0, " → 원점 {0} 뿐")
print("자유 변수가 없으므로 특별해도 없다. 정보를 하나도 안 잃는 행렬이다.")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)